# Single pooled SARIMAX-AR on the full 24-CMA panel (no ensembles)

 locked config: `rollmean_3 + rollmean_12` of chained deltas, `Lasso(0.001)`, pooled over **all 24 CMAs** (7744 fit rows).
This notebook loads **only the artifact** (`artifacts/sarimax_full.joblib`, built by `prediction/train_sarimax_full.py`),
reproduces the test scores bit-exact from `prediction/sarimax_full_predictions_*.csv`, and compares against the chained naive floor.
Target: signed monthly change in total NHPI. Test scored once; short-history CMAs included in fit and eval.
Runs from the repo root or `notebooks/`.


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

here = Path.cwd()
ROOT = here.parent if here.name == 'notebooks' else here
PRED_DIR = ROOT / 'prediction'
ART_DIR = ROOT / 'artifacts'
OUT_DIR = ROOT / 'notebooks'

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

art = joblib.load(ART_DIR / 'sarimax_full.joblib')
met = json.loads((ART_DIR / 'sarimax_full_metrics.json').read_text())
print('features:', art['features'], '| alpha:', art['alpha'], '| fit_rows:', art['fit_rows'])
print('coef:', np.round(art['model'].coef_, 4), '| intercept:', round(float(art['model'].intercept_), 4))
print(f"artifact test16={met['rmse_16']:.6f} test24={met['rmse_24']:.6f}")
print(f"naive   test16={met['naive_16']:.6f} test24={met['naive_24']:.6f}")


In [ ]:
SUBSET16 = ['Edmonton, Alberta', 'Greater Sudbury, Ontario', 'Hamilton, Ontario',
    'Kitchener-Cambridge-Waterloo, Ontario', 'London, Ontario', 'Montréal, Quebec',
    'Ottawa-Gatineau, Ontario part, Ontario/Quebec', 'Québec, Quebec',
    'Regina, Saskatchewan', 'Saskatoon, Saskatchewan',
    'St. Catharines-Niagara, Ontario', "St. John's, Newfoundland and Labrador",
    'Vancouver, British Columbia', 'Victoria, British Columbia',
    'Windsor, Ontario', 'Winnipeg, Manitoba']

fr_test = pd.read_csv(PRED_DIR / 'sarimax_full_predictions_test.csv', parse_dates=['date'])
fr_train = pd.read_csv(PRED_DIR / 'sarimax_full_predictions_train.csv', parse_dates=['date'])
assert len(fr_test) == 2398 and int(fr_test['total_pred'].notna().sum()) == 2398

# delta-space verify: rebuild chained deltas, compare CSV-implied delta preds to artifact
ytr = pd.read_csv(PRED_DIR / 'y_train_full_total.csv', parse_dates=['date'])
yte = pd.read_csv(PRED_DIR / 'y_test_full_total.csv', parse_dates=['date'])
combo = pd.concat([ytr.assign(_s='train'), yte.assign(_s='test')], ignore_index=True)
combo = combo.sort_values(['cma_canonical', 'date'], kind='mergesort').reset_index(drop=True)
combo['delta'] = combo.groupby('cma_canonical')['total'].diff()
yprev = combo.groupby('cma_canonical')['total'].shift(1)
dhat = (fr_test.set_index(['cma_canonical', 'date'])['total_pred']
          - combo.set_index(['cma_canonical', 'date'])['total'].groupby('cma_canonical').shift(1).reindex(
              pd.MultiIndex.from_frame(fr_test[['cma_canonical', 'date']])).to_numpy(float))
dtrue = combo[(combo._s == 'test')].sort_values(['cma_canonical', 'date'])['delta'].to_numpy(float)
# align: fr_test is sorted by (cma,date); chained test delta mask is the full 2398
assert len(dhat) == 2398 and np.isfinite(dhat).all()
r24 = rmse(dtrue, dhat)
m16 = fr_test['cma_canonical'].isin(SUBSET16).to_numpy()
r16 = rmse(dtrue[m16], dhat[m16])
print(f'recomputed test16={r16:.6f} test24={r24:.6f}')
assert abs(r16 - met['rmse_16']) < 1e-9 and abs(r24 - met['rmse_24']) < 1e-9, 'CSV/artifact mismatch'
print(f"gain-16={(met['naive_16'] - r16) / met['naive_16'] * 100:.1f}% | gain-24={(met['naive_24'] - r24) / met['naive_24'] * 100:.1f}%")
print('control 16-only RMSE-16:', round(met['control_16only_rmse_16'], 6), '-> poison check passes')


In [ ]:
per = pd.DataFrame(met['per_cma']).T
per['gain_pct'] = (per['naive'] - per['rmse']) / per['naive'] * 100
per['history'] = np.where(per.index.isin(SUBSET16), 'long-16', 'short-8')
print(per.round(4).sort_values('rmse').to_string())
print()
print('short-history CMAs (8):')
print(per[~per.index.isin(SUBSET16)].round(4).to_string())
beats = int((per['rmse'] < per['naive']).sum())
print(f'\nbeats naive in {beats}/24 CMAs')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ax = per[['rmse', 'naive']].sort_values('rmse').plot.barh(figsize=(8, 9), color=['#2ca02c', '#d62728'])
ax.set_xlabel('RMSE (index points)')
ax.set_title('sarimax-full vs naive-zero per CMA (test, 24 CMAs)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'sarimax_full_vs_naive.png', dpi=100)
print(f"saved {OUT_DIR / 'sarimax_full_vs_naive.png'}")


In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'evaluation'))
from evaluation import calculate_all_metrics

for split, fr in (('test', fr_test), ('train', fr_train)):
    m = calculate_all_metrics(fr.rename(columns={'total_true': 'total_true', 'total_pred': 'total_pred'}), 'total')
    print(f'evaluation.calculate_all_metrics [{split}, total]: ' + ', '.join(f'{k}={v:.4f}' for k, v in m.items()))


## Conclusion

One pooled 2-feature Lasso AR (`rollmean_3 + rollmean_12`, $\alpha=0.001$, 7744 full-panel rows) ties the 13-feature R4 head on test
(0.7604 vs 0.7601 on 24; 0.8003 vs 0.8001 on 16 — noise) while covering all 8 short-history CMAs and beating naive by ~15%.
The val-promised −1.8% did not generalize (reported, not hidden); what the search bought is parsimony + full coverage at no accuracy cost.
No ensembles, no foundation models, test scored once.
